# Tutorial 4, Q2b: both factories planned together

Release after the practical.

**The problem.** One model for the whole company: `x[0]` = standard at A ($s_1$), `x[1]` = deluxe at A ($d_1$),
`x[2]` = standard at B ($s_2$), `x[3]` = deluxe at B ($d_2$). Maximise $10s_1 + 15d_1 + 10s_2 + 15d_2$ subject to
A's grinding $4s_1 + 2d_1 \le 80$ and polishing $2s_1 + 5d_1 \le 60$, B's grinding $5s_2 + 3d_2 \le 60$ and
polishing $5s_2 + 6d_2 \le 75$, and **one shared** raw-material limit $4(s_1 + d_1 + s_2 + d_2) \le 120$ kg;
all whole numbers $\ge 0$.

**How to run it.** Run the first code cell first (it installs Pyomo and the HiGHS solver; the first time takes about a minute), then *Run → Run All Cells*, or run the cells one by one while you read.

**Expected result.** $s_1 = 10$, $d_1 = 8$, $s_2 = 0$, $d_2 = 12$: profit **£400**, against £220 + £165 = £385
for the two separate plans. Factory A uses 72 kg of raw material and factory B 48 kg.

*Original script: `5 Source Files/Python/tutorial_q2b_factory_combined.py`. Runs in the browser with HiGHS instead of GLPK; the model and answers are unchanged.*

In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

Load Pyomo's modelling tools.

In [ ]:
from pyomo.environ import *

A *concrete model* is the container for the problem.

In [ ]:
# Create a Concrete Model
model = ConcreteModel()

Four whole-number variables, in the order $s_1, d_1, s_2, d_2$ (factory A first, then B).

In [ ]:
# Integer decision variables: x[0] = s1, x[1] = d1 (factory A), x[2] = s2, x[3] = d2 (factory B)
model.x = Var(range(4), domain=NonNegativeIntegers)

Profit per unit for the four variables.

In [ ]:
# Profit per unit (£) for s1, d1, s2, d2
profit = [10, 15, 10, 15]

Resource use per unit. Each factory's machines only work on its own products, so A's lists have 0 for B's variables and B's lists have 0 for A's. Raw material is shared, so every product uses 4 kg.

In [ ]:
# Resource use per unit for [s1, d1, s2, d2]; 0 = not made at that factory
grind_a = [4, 2, 0, 0]
polish_a = [2, 5, 0, 0]
grind_b = [0, 0, 5, 3]
polish_b = [0, 0, 5, 6]
raw = [4, 4, 4, 4]

The objective: total profit of both factories, maximised.

In [ ]:
# Objective: maximise the total profit of both factories
model.obj = Objective(expr=sum(profit[i] * model.x[i] for i in model.x), sense=maximize)

Two machine constraints per factory, and one raw-material constraint for the whole company: the model now decides how to split the 120 kg.

In [ ]:
# Define the inequality constraints
model.grinding_cap_factorya = Constraint(expr=sum(grind_a[i] * model.x[i] for i in model.x) <= 80)  # factory A grinding: 4s1 + 2d1 <= 80 hours
model.polishing_cap_factorya = Constraint(expr=sum(polish_a[i] * model.x[i] for i in model.x) <= 60)  # factory A polishing: 2s1 + 5d1 <= 60 hours
model.grinding_cap_factoryb = Constraint(expr=sum(grind_b[i] * model.x[i] for i in model.x) <= 60)  # factory B grinding: 5s2 + 3d2 <= 60 hours
model.polishing_cap_factoryb = Constraint(expr=sum(polish_b[i] * model.x[i] for i in model.x) <= 75)  # factory B polishing: 5s2 + 6d2 <= 75 hours
model.raw_cap = Constraint(expr=sum(raw[i] * model.x[i] for i in model.x) <= 120)  # shared raw material: 4(s1 + d1 + s2 + d2) <= 120 kg

Solve with HiGHS; `tee=True` prints the solver's log. Near the end it should say the model status is *Optimal*.

In [ ]:
# Solve the optimization problem
solver = SolverFactory('appsi_highs')  # HiGHS runs in the browser (the original script used GLPK)
solver.solve(model, tee=True)

`pprint()` prints the whole model with the data filled in: check that it matches the formulation above.

In [ ]:
# Print the model
model.pprint()

`display()` prints the values: the number of each product, the profit, and how much of each resource is used next to its limit.

In [ ]:
# Print the results
model.display()

Raw material used by each variable. The printed list has **four** numbers, one per variable ($s_1, d_1, s_2, d_2$), although the label says *for each factory*: add the first two for factory A and the last two for factory B.

In [ ]:
# Raw material used per variable [s1, d1, s2, d2]: factory A = first two, B = last two
used_raw = [raw[i] * value(model.x[i]) for i in model.x]
print("Used raw material for each factory:", used_raw)

## What do you notice?

How much raw material does each factory use, and why is the company's profit higher than £385?

<details><summary>Answer</summary>

Factory A uses $40 + 32 = 72$ kg and factory B $0 + 48 = 48$ kg: a 72:48 split instead of the manual 75:45.
Factory A could not use its last 3 kg (polishing is binding there), while factory B had spare machine hours for one
more deluxe unit (4 kg, £15). Moving 3 kg from A to B lets B make 12 deluxe instead of 11: £400 instead of £385.
Planning parts of a system separately can miss a better plan for the whole. With fractional units allowed the
company could make £404.17.
</details>